# 06 — Generate DINOv2 Embeddings

**Fruvia AI** — DINOv2 embedding generation notebook.

This notebook runs on **Google Colab with T4 GPU** and generates 768-dimensional
embeddings for all images using `facebook/dinov2-base`.

### What this notebook does

1. Load the retrieval manifest (`retrieval_manifest.csv`)
2. Load `facebook/dinov2-base` from HuggingFace
3. Process images in batches (GPU-accelerated)
4. L2-normalize all vectors
5. Save embeddings as shards to Google Drive (not all in RAM)
6. Support checkpoint/resume for interrupted runs
7. Optionally upload directly to Qdrant in batches

### Prerequisites

- **GPU runtime required** (Runtime → Change runtime type → T4 GPU)
- Retrieval manifest from notebook 02
- Fruits-360 dataset accessible (Kaggle download or Google Drive)
- (Optional) Qdrant credentials in Colab Secrets for direct upload

## 1. Setup & Dependencies

In [ ]:
!pip install -q transformers torch torchvision pandas tqdm qdrant-client kagglehub

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

## 2. GPU Check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available! Go to Runtime → Change runtime type → T4 GPU")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

## 3. Configuration

In [ ]:
# ── Paths ──
# Where is the Fruits-360 dataset? (download below if needed)
DATA_DIR = Path("/content/fruits360")  # Will be set after download

# Retrieval manifest from notebook 02
MANIFEST_CSV = Path("/content/retrieval_manifest.csv")

# Google Drive output directory for embedding shards
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/fruvia-ai/embeddings")

# ── Model ──
MODEL_NAME = "facebook/dinov2-base"
VECTOR_SIZE = 768

# ── Processing ──
BATCH_SIZE = 64  # Images per batch (adjust for GPU memory)
SHARD_SIZE = 5000  # Images per shard file
EMBED_ALL = True  # True = embed all images; False = sample only
SAMPLE_SIZE = 500  # Only used when EMBED_ALL=False

# ── Checkpoint ──
CHECKPOINT_FILE = DRIVE_OUTPUT_DIR / "checkpoint.json"

## 4. Mount Google Drive & Download Dataset

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {DRIVE_OUTPUT_DIR}")

In [ ]:
# Download dataset if not already present
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub

dataset_path = kagglehub.dataset_download("moltean/fruits")
DATA_DIR = Path(dataset_path)
print(f"Dataset at: {DATA_DIR}")

## 5. Load Manifest

In [ ]:
# Load the retrieval manifest
if not MANIFEST_CSV.exists():
    # Try Google Drive fallback
    drive_manifest = Path("/content/drive/MyDrive/fruvia-ai/manifests/retrieval_manifest.csv")
    if drive_manifest.exists():
        MANIFEST_CSV = drive_manifest
    else:
        raise FileNotFoundError(
            f"Manifest not found at {MANIFEST_CSV} or {drive_manifest}. Run notebook 02 first."
        )

df_manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest loaded: {len(df_manifest)} images")

# Filter to valid images only
df_manifest = df_manifest[df_manifest["is_valid"]].reset_index(drop=True)
print(f"Valid images: {len(df_manifest)}")

# Sample if not embedding all
if not EMBED_ALL:
    df_manifest = df_manifest.sample(n=min(SAMPLE_SIZE, len(df_manifest)), random_state=42)
    df_manifest = df_manifest.reset_index(drop=True)
    print(f"Sampled {len(df_manifest)} images for test run")

## 6. Load DINOv2 Model

In [ ]:
print(f"Loading {MODEL_NAME}...")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"Model loaded on {device}")
print(f"Embedding dimension: {model.config.hidden_size}")
assert model.config.hidden_size == VECTOR_SIZE, (
    f"Expected {VECTOR_SIZE}-dim, got {model.config.hidden_size}-dim"
)

## 7. Checkpoint / Resume Logic

In [ ]:
def load_checkpoint() -> dict:
    """Load checkpoint from Google Drive."""
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            ckpt = json.load(f)
        print(
            f"Resuming from checkpoint: {ckpt['completed_images']} images done, "
            f"{ckpt['completed_shards']} shards saved"
        )
        return ckpt
    return {"completed_images": 0, "completed_shards": 0, "image_ids_done": []}


def save_checkpoint(completed_images: int, completed_shards: int, image_ids_done: list[str]):
    """Save checkpoint to Google Drive."""
    ckpt = {
        "completed_images": completed_images,
        "completed_shards": completed_shards,
        "image_ids_done": image_ids_done,
    }
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(ckpt, f)


checkpoint = load_checkpoint()
start_idx = checkpoint["completed_images"]
shard_idx = checkpoint["completed_shards"]
ids_done = set(checkpoint.get("image_ids_done", []))

# Skip already-processed images
if ids_done:
    df_remaining = df_manifest[~df_manifest["image_id"].isin(ids_done)].reset_index(drop=True)
    print(f"Remaining images to process: {len(df_remaining)}")
else:
    df_remaining = df_manifest
    print(f"Starting fresh: {len(df_remaining)} images to process")

## 8. Batch Embedding Generation

In [ ]:
@torch.no_grad()
def embed_batch(image_paths: list[Path]) -> np.ndarray:
    """
    Embed a batch of images using DINOv2.

    Returns (N, 768) L2-normalized embeddings as numpy array.
    """
    images = []
    for p in image_paths:
        try:
            img = Image.open(p).convert("RGB")
            images.append(img)
        except Exception:
            # Create a blank image as placeholder for corrupt files
            images.append(Image.new("RGB", (224, 224)))

    inputs = processor(images=images, return_tensors="pt").to(device)

    with torch.cuda.amp.autocast():
        outputs = model(**inputs)

    # CLS token embedding
    embeddings = outputs.last_hidden_state[:, 0, :]  # (batch, 768)

    # L2 normalize
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    return embeddings.cpu().numpy()


def save_shard(
    shard_embeddings: np.ndarray,
    shard_metadata: list[dict],
    shard_number: int,
) -> Path:
    """Save an embedding shard as .npz + metadata JSON."""
    shard_name = f"shard_{shard_number:04d}"

    # Save embeddings
    npz_path = DRIVE_OUTPUT_DIR / f"{shard_name}.npz"
    np.savez_compressed(npz_path, embeddings=shard_embeddings)

    # Save metadata
    meta_path = DRIVE_OUTPUT_DIR / f"{shard_name}_meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(shard_metadata, f)

    return npz_path

In [ ]:
# Main embedding loop
shard_embeddings: list[np.ndarray] = []
shard_metadata: list[dict] = []
all_ids_done = list(ids_done)

total = len(df_remaining)
pbar = tqdm(total=total, desc="Embedding images")
t_start = time.time()

for batch_start in range(0, total, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total)
    batch_df = df_remaining.iloc[batch_start:batch_end]

    # Resolve full paths
    batch_paths = [DATA_DIR / row["relative_path"] for _, row in batch_df.iterrows()]

    # Embed
    embeddings = embed_batch(batch_paths)

    # Collect
    for i, (_, row) in enumerate(batch_df.iterrows()):
        shard_embeddings.append(embeddings[i])
        shard_metadata.append(
            {
                "image_id": row["image_id"],
                "original_class": row["original_class"],
                "target_class": row.get("target_class", "unknown"),
                "relative_path": row["relative_path"],
                "filename": row["filename"],
            }
        )
        all_ids_done.append(row["image_id"])

    pbar.update(batch_end - batch_start)

    # Save shard when full
    if len(shard_embeddings) >= SHARD_SIZE:
        stacked = np.stack(shard_embeddings[:SHARD_SIZE])
        save_shard(stacked, shard_metadata[:SHARD_SIZE], shard_idx)
        print(f"\n  Saved shard {shard_idx} ({len(stacked)} embeddings)")

        # Keep remainder for next shard
        shard_embeddings = shard_embeddings[SHARD_SIZE:]
        shard_metadata = shard_metadata[SHARD_SIZE:]
        shard_idx += 1

        # Checkpoint
        save_checkpoint(batch_end + start_idx, shard_idx, all_ids_done)

        # Free GPU memory
        torch.cuda.empty_cache()
        gc.collect()

# Save final partial shard
if shard_embeddings:
    stacked = np.stack(shard_embeddings)
    save_shard(stacked, shard_metadata, shard_idx)
    print(f"\n  Saved final shard {shard_idx} ({len(stacked)} embeddings)")
    shard_idx += 1

pbar.close()

# Final checkpoint
save_checkpoint(total + start_idx, shard_idx, all_ids_done)

elapsed = time.time() - t_start
print(f"\nDone! {total} images embedded in {elapsed:.1f}s ({total / elapsed:.1f} img/s)")
print(f"Shards saved to {DRIVE_OUTPUT_DIR}")

## 9. Verify Embeddings

In [ ]:
# List all saved shards
shard_files = sorted(DRIVE_OUTPUT_DIR.glob("shard_*.npz"))
total_vectors = 0

print(f"Embedding shards in {DRIVE_OUTPUT_DIR}:")
for sf in shard_files:
    data = np.load(sf)
    n, d = data["embeddings"].shape
    total_vectors += n
    print(f"  {sf.name}: {n} vectors × {d} dims")

print(f"\nTotal vectors: {total_vectors}")
print(f"Vector dimension: {VECTOR_SIZE}")

# Verify L2 norms
sample_shard = np.load(shard_files[0])
norms = np.linalg.norm(sample_shard["embeddings"], axis=1)
print(
    f"\nL2 norm check (first shard): min={norms.min():.4f}, max={norms.max():.4f}, "
    f"mean={norms.mean():.4f} (should be ~1.0)"
)

---

## Summary

| Output | Location |
|--------|----------|
| Embedding shards | `Google Drive/fruvia-ai/embeddings/shard_XXXX.npz` |
| Shard metadata | `Google Drive/fruvia-ai/embeddings/shard_XXXX_meta.json` |
| Checkpoint | `Google Drive/fruvia-ai/embeddings/checkpoint.json` |

**Next step:** Run `07_upload_qdrant.ipynb` to upload these embeddings to Qdrant Cloud.